In [10]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error as mse 
from sklearn.metrics import mean_absolute_error as mae
from math import sqrt

from statsforecast.models import AutoARIMA, AutoETS
from statsforecast import StatsForecast
from prophet import Prophet

data = pd.read_csv('C:/Users/LENOVO/Documents/python_data_analytics_course/Pod 2 Project/rainfall_2000_2026.csv')
temp_data = pd.read_csv('C:/Users/LENOVO/Documents/python_data_analytics_course/Pod 2 Project/sarima_app/utils/data/Temperature/era_5_2000_2026.csv')

DATA CLEANING, PREPROCESSING AND FEATURE ENGINEERING

In [ ]:
#Convert data to datetime object and create new columns for month and year

data['date'] = pd.to_datetime(data['date'])
data['year'] = data['date'].dt.year
data['logged_tp'] = np.sqrt(data['tp'])

X = data[['logged_tp', 'year', 'date']].copy()
X['unique_id'] = 'weather'
rain_train = X[X['year'] <= 2023].copy()
rain_train.drop(columns=['year'], axis = 1, inplace= True)
rain_test = X[X['year'] > 2023].copy()
rain_test.drop(columns=['year'], axis = 1, inplace= True)

    
rain_train = rain_train.rename(columns={
    'date': 'ds',
    'logged_tp': 'y'
})
      
rain_test = rain_test.rename(columns={
    'date': 'ds',
    'logged_tp': 'y'
})

rain_train = rain_train.sort_values(['unique_id', 'ds'])
rain_test = rain_test.sort_values(['unique_id', 'ds'])

freq = pd.infer_freq(rain_test['ds'])

model = AutoARIMA(season_length=12, alias = 'SARIMA')
sf = StatsForecast(models = [model], freq = freq)
sf.fit(rain_train)
pred = sf.predict(h = 6)


pred['actual'] = rain_test['y'].iloc[:6].values
pred['diff'] = pred['actual'] - pred['SARIMA']
pred




,unique_id,ds,SARIMA,actual,diff
0,weather,2024-01-01,0.536580,0.102623,-0.433957
1,weather,2024-02-01,1.113423,0.768672,-0.344751
2,weather,2024-03-01,2.989274,1.804929,-1.184345
3,weather,2024-04-01,4.677467,4.385715,-0.291751
4,weather,2024-05-01,8.070291,3.944753,-4.125538
5,weather,2024-06-01,11.462614,6.873516,-4.589098


In [11]:
#Convert data to datetime object and create new columns for month and year

data['date'] = pd.to_datetime(data['date'])
data['year'] = data['date'].dt.year
data['logged_tp'] = np.sqrt(data['tp'])

X = data[['logged_tp', 'year', 'date']].copy()
X['unique_id'] = 'weather'
rain_train = X[X['year'] <= 2023].copy()
rain_train.drop(columns=['year'], axis = 1, inplace= True)
rain_test = X[X['year'] > 2023].copy()
rain_test.drop(columns=['year'], axis = 1, inplace= True)

    
rain_train = rain_train.rename(columns={
    'date': 'ds',
    'logged_tp': 'y'
})
      
rain_test = rain_test.rename(columns={
    'date': 'ds',
    'logged_tp': 'y'
})

rain_train = rain_train.sort_values(['unique_id', 'ds'])
rain_test = rain_test.sort_values(['unique_id', 'ds'])

freq = pd.infer_freq(rain_test['ds'])

model = AutoARIMA(season_length=12, alias = 'SARIMA')
sf = StatsForecast(models = [model], freq = freq)
sf.fit(rain_train)
pred = sf.predict(h = 6)

pred = pred['SARIMA'].apply(lambda x: np.square(x))

pred




0      0.287918
1      1.239711
2      8.935759
3     21.878695
4     65.129603
5    131.391512
Name: SARIMA, dtype: float64

In [26]:
mae(np.square(pred['SARIMA']), np.square(pred['actual']))

23.82720709613257

PROPHET

In [ ]:
#Convert data to datetime object and create new columns for month and year

# data['date'] = pd.to_datetime(data['date'])
# data['year'] = data['date'].dt.year

# data.drop('extreme_rain', axis = 1, inplace = True)

X = data[['logged_tp', 'year', 'date']].copy()
rain_train = X[X['year'] <= 2023].copy()
rain_train.drop(columns=['year'], axis = 1, inplace= True)
rain_test = X[X['year'] > 2023].copy()
rain_test.drop(columns=['year'], axis = 1, inplace= True)

    
rain_train = rain_train.rename(columns={
    'date': 'ds',
    'logged_tp': 'y'
})
      
rain_test = rain_test.rename(columns={
    'date': 'ds',
    'logged_tp': 'y'
})

rain_train = rain_train.sort_values('ds')
rain_test = rain_test.sort_values( 'ds')

freq = pd.infer_freq(rain_test['ds'])

model = Prophet(
    yearly_seasonality= True,
    weekly_seasonality = False,
    daily_seasonality = False,
    seasonality_mode = 'additive'
)
model.fit(rain_train)

future = model.make_future_dataframe(periods = 25, freq = 'M')
forecast = model.predict(future)

forecast



19:31:54 - cmdstanpy - INFO - Chain [1] start processing
19:31:54 - cmdstanpy - INFO - Chain [1] done processing


In [30]:
#Convert data to datetime object and create new columns for month and year

temp_data['date'] = pd.to_datetime(data['date'])
temp_data['year'] = data['date'].dt.year

# data.drop('extreme_rain', axis = 1, inplace = True)

X = temp_data[['tp', 'year', 'date']].copy()
rain_train = X[X['year'] <= 2023].copy()
rain_train.drop(columns=['year'], axis = 1, inplace= True)
rain_test = X[X['year'] > 2023].copy()
rain_test.drop(columns=['year'], axis = 1, inplace= True)

    
rain_train = rain_train.rename(columns={
    'date': 'ds',
    'tp': 'y'
})
      
rain_test = rain_test.rename(columns={
    'date': 'ds',
    'tp': 'y'
})

rain_train = rain_train.sort_values('ds')
rain_test = rain_test.sort_values( 'ds')

freq = pd.infer_freq(rain_test['ds'])

model = Prophet(
    yearly_seasonality= True,
    weekly_seasonality = False,
    daily_seasonality = False,
    seasonality_mode = 'additive'
)
model.fit(rain_train)

future = model.make_future_dataframe(periods = 25, freq = 'M')
forecast = model.predict(future)

forecast



20:06:14 - cmdstanpy - INFO - Chain [1] start processing
20:06:14 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\LENOVO\anaconda3\envs\python_course\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(


,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2000-01-01,2.524432,-1.593604,1.338522,2.524432,2.524432,-2.635609,-2.635609,-2.635609,-2.635609,-2.635609,-2.635609,0.0,0.0,0.0,-0.111177
1,2000-02-01,2.524728,-1.404773,1.388728,2.524728,2.524728,-2.517369,-2.517369,-2.517369,-2.517369,-2.517369,-2.517369,0.0,0.0,0.0,0.007359
2,2000-03-01,2.525005,-1.244538,1.458441,2.525005,2.525005,-2.401384,-2.401384,-2.401384,-2.401384,-2.401384,-2.401384,0.0,0.0,0.0,0.123621
3,2000-04-01,2.525300,-0.348162,2.491687,2.525300,2.525300,-1.425762,-1.425762,-1.425762,-1.425762,-1.425762,-1.425762,0.0,0.0,0.0,1.099538
4,2000-05-01,2.525587,0.854102,3.761645,2.525587,2.525587,-0.219843,-0.219843,-0.219843,-0.219843,-0.219843,-0.219843,0.0,0.0,0.0,2.305744
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
308,2025-08-31,2.220267,4.210717,6.911343,2.216320,2.223947,3.369736,3.369736,3.369736,3.369736,3.369736,3.369736,0.0,0.0,0.0,5.590003
309,2025-09-30,2.216973,2.436020,5.199237,2.212725,2.220932,1.572824,1.572824,1.572824,1.572824,1.572824,1.572824,0.0,0.0,0.0,3.789798
310,2025-10-31,2.213570,-1.472930,1.185788,2.208996,2.217781,-2.425384,-2.425384,-2.425384,-2.425384,-2.425384,-2.425384,0.0,0.0,0.0,-0.211814
311,2025-11-30,2.210276,-2.121984,0.694802,2.205232,2.214804,-2.956469,-2.956469,-2.956469,-2.956469,-2.956469,-2.956469,0.0,0.0,0.0,-0.746193


In [31]:
forecast_for_app = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(24)
eval_df = forecast_for_app.merge(rain_test[['ds', 'y']], on='ds')

forecast_for_app['actual'] = rain_test['y'].values

forecast_for_app


,ds,yhat,yhat_lower,yhat_upper,actual
289,2024-01-31,-0.417807,-1.765271,0.953259,0.000329
290,2024-02-29,0.299631,-1.171313,1.865280,0.018464
291,2024-03-31,0.855710,-0.591079,2.275804,0.101805
292,2024-04-30,2.736299,1.338261,4.128777,0.601078
293,2024-05-31,3.779248,2.380619,5.206526,0.486284
294,2024-06-30,4.781475,3.355086,6.186355,1.476413
295,2024-07-31,8.187023,6.761420,9.533282,5.605449
296,2024-08-31,5.963646,4.560533,7.282509,4.680771
297,2024-09-30,3.369808,1.890416,4.706451,6.858297
298,2024-10-31,-0.158938,-1.486769,1.212565,3.695639


In [33]:
mae(forecast_for_app['actual'], forecast_for_app['yhat'])

1.6977854922445692

In [ ]:
rain_test.reset_index(inplace = True)
rain_test

In [ ]:
mae(np.square(rain_test['y'].iloc[:6]), np.square(pred['SARIMA']))

In [ ]:
y_naive = rain_test['y'].shift(12)   # same month last year
mae_naive = mae(np.square(rain_test['y'][12:]), np.square(y_naive[12:]))

mae_naive


In [ ]:
ets = AutoETS(season_length=12, alias = 'ETS')
sf_ets = StatsForecast(models = [ets], freq = freq)
sf_ets.fit(rain_train)
pred_ets = sf.predict(h = 24)

pred_ets.head()

In [ ]:
X.describe()

In [ ]:
def run_forecast(data, year, freq = 'MS'):
    """
    to run a forecast
    """

    import pandas as pd
    from statsforecast.models import AutoARIMA, AutoETS
    from statsforecast import StatsForecast

    data['date'] = pd.to_datetime(data['date'])
    data['year'] = data['date'].dt.year
    X = data[['tp', 'year', 'date']].copy()
    X['unique_id'] = 'rainfall series'
    rain_train = X[X['year'] <= 2023].copy()
    rain_train.drop(columns=['year'], axis = 1, inplace= True)
    rain_test = X[X['year'] > 2023].copy()
    rain_test.drop(columns=['year'], axis = 1, inplace= True)

    rain_train = rain_train.rename(columns={
        'date': 'ds',
        'tp': 'y'
    })
      
    rain_test = rain_test.rename(columns={
        'date': 'ds',
        'tp': 'y'
    })

    rain_train = rain_train.sort_values(['unique_id', 'ds'])
    rain_test = rain_test.sort_values(['unique_id', 'ds'])

    model = AutoARIMA(season_length=12, alias = 'SARIMA')
    sf = StatsForecast(models = [model], freq = freq)
    sf.fit(rain_train)

    last_year = data['date'].dt.year.max()
    steps = (year - last_year) * 12
    pred = sf.predict(h = steps)

    return pred['SARIMA'].sum()



    
    
    

In [ ]:
run_forecast(data, 2026)

In [ ]:
import pickle

In [ ]:
data['rain_lag1'] = data['tp'].shift(1)
data['rain_lag2'] = data['tp'].shift(2)

data['temp_lag1'] = data['t2m'].shift(1)
data['temp_lag2'] = data['t2m'].shift(2)

data['rain_rolling3'] = data['tp'].rolling(3).sum()
data['temp_rolling3'] = data['t2m'].rolling(3).mean()

In [ ]:
import pickle

In [ ]:
with open("sarima_model.pkl", "wb") as f:
    pickle.dump(sf, f)

In [ ]:
loaded_model = pickle.load(open('sarima_model.pkl', 'rb'))

In [ ]:
def run_forecast(data, year, freq="MS"):
    """
    Run rainfall forecast using StatsForecast AutoARIMA
    """

    import pandas as pd
    from statsforecast.models import AutoARIMA
    from statsforecast import StatsForecast

    data = data.copy()
    data["date"] = pd.to_datetime(data["date"])
    data["year"] = data["date"].dt.year

    X = data[["tp", "date"]].copy()
    X["unique_id"] = "rainfall_series"

    rain_train = X[data["year"] <= 2023].copy()

    rain_train = rain_train.rename(
        columns={
            "date": "ds",
            "tp": "y"
        }
    )

    rain_train = rain_train.sort_values(["unique_id", "ds"])

    model = AutoARIMA(season_length=12, alias="SARIMA")
    sf = StatsForecast(models=[model], freq=freq)

    sf.fit(rain_train)

    last_year = data["year"].max()
    steps = (year - last_year) * 12

    pred = sf.predict(h=steps)
    return pred["SARIMA"].sum()

In [ ]:


loaded_model.fit(rain_train)

pred = sf.predict(h=24)

In [ ]:
from sklearn.metrics import mean_absolute_error as mae 
from sklearn.metrics import mean_squared_error as mse

In [ ]:
mae()

In [7]:
import timesfm
import torch

# Initialize model
# Initialize TimesFM
tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend="gpu",
        per_core_batch_size=32,
        horizon_len=128,
        num_layers=50,
        context_len=2048,
        use_positional_embedding=False,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id="google/timesfm-2.0-500m-pytorch"),
)

print("Model loaded")


AttributeError: module 'timesfm' has no attribute 'TimesFm'

In [5]:
help(timesfm)

Help on package timesfm:

NAME
    timesfm - TimesFM API.

PACKAGE CONTENTS
    configs
    flax (package)
    torch (package)

SUBMODULES
    timesfm_2p5

FILE
    c:\users\lenovo\documents\python_data_analytics_course\pod 2 project\timesfm\src\timesfm\__init__.py




In [2]:
pip install torch torchvision torchaudio

   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/113.7 MB 3.0 MB/s eta 0:00:38
    --------------------------------------- 1.8/113.7 MB 3.6 MB/s eta 0:00:32
   - -------------------------------------- 2.9/113.7 MB 3.9 MB/s eta 0:00:29
   - -------------------------------------- 3.7/113.7 MB 4.1 MB/s eta 0:00:27
   - -------------------------------------- 4.7/113.7 MB 4.1 MB/s eta 0:00:27
   -- ------------------------------------- 5.8/113.7 MB 4.3 MB/s eta 0:00:26
   -- ------------------------------------- 7.1/113.7 MB 4.4 MB/s eta 0:00:24
   -- ------------------------------------- 8.1/113.7 MB 4.6 MB/s eta 0:00:24
   --- ------------------------------------ 9.2/113.7 MB 4.7 MB/s eta 0:00:23
   --- ------------------------------------ 10.0/113.7 MB 4.7 MB/s eta 0:00:23
   --- ------------------------------------ 11.0/113.7 MB 4.6 MB/s eta 0:00:2